# DRL usage example

In [1]:
from sinergym.utils.callbacks import LoggerEvalCallback
from sinergym.utils.rewards import *
from sinergym.utils.wrappers import LoggerWrapper
from datetime import datetime
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3.common.vec_env import DummyVecEnv

import sys

#add elhogym to the PYTHONPATH
sys.path.insert(0, '/workspaces/elizabeth-homes/exp/hannes')
#load environment definition from input folder
import input


/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/envs/registration.py:216: UserWarning: WARN: Overriding environment Eplus-demo-v1
  logger.warn("Overriding environment {}".format(id))


In [2]:
environment = "Eplus-1storeytest-v2"
episodes = 10
experiment_date = datetime.today().strftime('%Y-%m-%d %H:%M')

# register run name
name = F"DQN-{environment}-episodes_{episodes}({experiment_date})"

In [3]:
env = gym.make(environment)

no file found at given path, content will be considered as empty (GBR_ENG_London.Wea.Ctr-St.James.Park.037700_TMYx.2004-2018.rain)
[2022-09-28 11:55:30,352] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf ExternalInterface object if it is not present...
[2022-09-28 11:55:30,353] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf Site:Location and SizingPeriod:DesignDay(s) to weather and ddy file...
[2022-09-28 11:55:30,362] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf OutPut:Variable and variables XML tree model for BVCTB connection.
[2022-09-28 11:55:30,363] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Setting up extra configuration in building model if exists...
[2022-09-28 11:55:30,364] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Setting up action definition in building model if exists...
/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


In [4]:
env = LoggerWrapper(env)

In [5]:
model = PPO('MlpPolicy', env, verbose=1,tensorboard_log="./drl_Eplus-1storeytest-v2_tensorboard/")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Now we need to calculate the number of timesteps of each episode for the evaluation.

In [6]:
n_timesteps_episode = env.simulator._eplus_one_epi_len / \
                      env.simulator._eplus_run_stepsize


Now we need to create a vectorized wrapper for the environment because the callbacks we are going to use require a vector.

In [7]:
env_vec = DummyVecEnv([lambda: env])

Sinergym callbacks can throw errors. I think it expects their standard form of rewards.

In [10]:
""" callbacks = []

# Set up Evaluation and saving best model
eval_callback = LoggerEvalCallback(
    env_vec,
    best_model_save_path='best_model/' + name + '/',
    log_path='best_model/' + name + '/',
    eval_freq=n_timesteps_episode * 2,
    deterministic=True,
    render=False,
    n_eval_episodes=2)
callbacks.append(eval_callback)

callback = CallbackList(callbacks) """

This is the number of total time steps for the training.

In [8]:
episodes = 20
timesteps = episodes * n_timesteps_episode

Training the model

In [9]:
model.learn(
    total_timesteps=timesteps,
    log_interval=1)

[2022-09-28 11:55:49,634] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Creating new EnergyPlus simulation episode...
[2022-09-28 11:55:49,639] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus working directory is in /workspaces/elizabeth-homes/exp/hannes/run/DRL-example/Eplus-env-1storeytest-v2-res5/Eplus-env-sub_run1
2022-09-28 11:55:50.885349: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-09-28 11:55:50.885366: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Logging to ./drl_Eplus-1storeytest-v2_tensorboard/PPO_6
-----------------------------
| time/              |      |
|    fps             | 430  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 2048 |
-----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 607           |
|    iterations           | 2             |
|    time_elapsed         | 6             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 4.1618478e-09 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.42         |
|    explained_variance   | 0             |
|    learning_rate        | 0.0003        |
|    loss                 | 5.32e+14      |
|    n_updates            | 10            |
|    policy_gradient_loss | -5.19e-07     |
|    std                  | 1 

/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:223: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
[2022-09-28 11:57:17,352] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus episode completed successfully. 
[2022-09-28 11:57:17,354

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 8.76e+04      |
|    ep_rew_mean          | -9.19e+10     |
| time/                   |               |
|    fps                  | 994           |
|    iterations           | 43            |
|    time_elapsed         | 88            |
|    total_timesteps      | 88064         |
| train/                  |               |
|    approx_kl            | 3.5506673e-09 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.42         |
|    explained_variance   | -1.19e-07     |
|    learning_rate        | 0.0003        |
|    loss                 | 8.43e+14      |
|    n_updates            | 420           |
|    policy_gradient_loss | -1.6e-07      |
|    std                  | 1             |
|    value_loss           | 1.49e+15      |
-------------------------------------------
--------------------------------

[2022-09-28 11:58:44,442] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus episode completed successfully. 
[2022-09-28 11:58:44,443] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Creating new EnergyPlus simulation episode...
[2022-09-28 11:58:44,450] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus working directory is in /workspaces/elizabeth-homes/exp/hannes/run/DRL-example/Eplus-env-1storeytest-v2-res5/Eplus-env-sub_run3


...

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 8.76e+04     |
|    ep_rew_mean          | -9.1e+10     |
| time/                   |              |
|    fps                  | 1007         |
|    iterations           | 770          |
|    time_elapsed         | 1565         |
|    total_timesteps      | 1576960      |
| train/                  |              |
|    approx_kl            | 5.005859e-09 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -1.19e-07    |
|    learning_rate        | 0.0003       |
|    loss                 | 7.94e+14     |
|    n_updates            | 7690         |
|    policy_gradient_loss | -6.46e-07    |
|    std                  | 1            |
|    value_loss           | 1.59e+15     |
------------------------------------------
------------------------------------------
| rollout/ 

[2022-09-28 12:23:21,295] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus episode completed successfully. 
[2022-09-28 12:23:21,297] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Creating new EnergyPlus simulation episode...
[2022-09-28 12:23:21,330] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus working directory is in /workspaces/elizabeth-homes/exp/hannes/run/DRL-example/Eplus-env-1storeytest-v2-res5/Eplus-env-sub_run20


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 8.76e+04      |
|    ep_rew_mean          | -9.09e+10     |
| time/                   |               |
|    fps                  | 1007          |
|    iterations           | 813           |
|    time_elapsed         | 1652          |
|    total_timesteps      | 1665024       |
| train/                  |               |
|    approx_kl            | 5.7334546e-09 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.42         |
|    explained_variance   | -1.19e-07     |
|    learning_rate        | 0.0003        |
|    loss                 | 6.58e+14      |
|    n_updates            | 8120          |
|    policy_gradient_loss | -3.73e-07     |
|    std                  | 1             |
|    value_loss           | 1.32e+15      |
-------------------------------------------
--------------------------------

[2022-09-28 12:24:48,372] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus episode completed successfully. 
[2022-09-28 12:24:48,375] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Creating new EnergyPlus simulation episode...
[2022-09-28 12:24:48,407] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:EnergyPlus working directory is in /workspaces/elizabeth-homes/exp/hannes/run/DRL-example/Eplus-env-1storeytest-v2-res5/Eplus-env-sub_run21


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 8.76e+04     |
|    ep_rew_mean          | -9.09e+10    |
| time/                   |              |
|    fps                  | 1007         |
|    iterations           | 856          |
|    time_elapsed         | 1739         |
|    total_timesteps      | 1753088      |
| train/                  |              |
|    approx_kl            | 5.500624e-09 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -1.19e-07    |
|    learning_rate        | 0.0003       |
|    loss                 | 3.91e+14     |
|    n_updates            | 8550         |
|    policy_gradient_loss | -1.17e-06    |
|    std                  | 1            |
|    value_loss           | 8.36e+14     |
------------------------------------------


Now we save the current model.

In [11]:
model.save(env.simulator._env_working_dir_parent + '/' + name)

And as always, remember to close the environment.

In [12]:
env.close()

[2022-08-24 09:09:48,272] EPLUS_ENV_demo-v1_MainThread_ROOT INFO:EnergyPlus simulation closed successfully. 
